# Products Bronze to Silver Transformation

## Objective:

### Transform Product Information data from the Bronze layer into the Silver layer by applying data quality validations checks, business rule validations,and  before storing the cleansed data in ADLS Gen2.
### Source: Bronze (CSV)
### Destination: Silver (Parquet)
### Technologies: Azure Databricks, PySpark, ADLS Gen2, Parquet

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
display(
    dbutils.fs.ls("abfss://bronze@stretaillakehouse01.dfs.core.windows.net/products/")
)

path,name,size,modificationTime
abfss://bronze@stretaillakehouse01.dfs.core.windows.net/products/products_final.csv,products_final.csv,77035,1782333962000


In [0]:
display(
    spark.read
         .option("header", "true")
         .csv("abfss://bronze@stretaillakehouse01.dfs.core.windows.net/products/products_final.csv")
)

Product_ID,Product_Name,Category,Sub_Category,Brand,Unit_Price,Unit,Launch_Date
P0001,Wildcraft Backpack,Fashion,Wildcraft,Wildcraft,47511.43,Each,2024-01-22
P0002,Nike Sports T-Shirt,Fashion,Nike,Nike,34977.22,Each,2022-10-08
P0003,Prestige Pressure Cooker 5L,Home & Kitchen,Prestige,Prestige,49429.51,Set,2026-01-22
P0004,Tata Tea Gold,Grocery,Tata,Tata,24461.77,Litre,2023-02-19
P0005,Yoga Mat Pro,Sports & Fitness,Yoga,FitPro,43225.65,Each,2025-12-08
P0006,Milton Water Bottle,Home & Kitchen,Milton,Milton,13875.58,Set,2026-05-06
P0007,Boat Rockerz 450,Electronics,Boat,Boat,30877.01,Each,2023-10-09
P0008,Wildcraft Backpack,Fashion,Wildcraft,Wildcraft,15271.17,Pair,2024-08-22
P0009,Cello Storage Container,Home & Kitchen,Cello,Cello,7090.75,Set,2023-09-21
P0010,Studds Full Face Helmet,Vehicle Accessories,Studds,Studds,19247.52,Each,2024-09-22


In [0]:
product_schema = '''
Product_ID STRING,
Product_Name STRING,
Product_Category STRING,
Product_SubCategory STRING,
Product_Brand STRING,
Product_Price DOUBLE,
Product_Unit STRING,
Product_Launch_Date DATE
'''


In [0]:
display(product_schema)

'\nProduct_ID STRING,\nProduct_Name STRING,\nProduct_Category STRING,\nProduct_SubCategory STRING,\nProduct_Brand STRING,\nProduct_Price DOUBLE,\nProduct_Unit STRING,\nProduct_Launch_Date DATE\n'

In [0]:
df_products = (

    spark.read.schema(product_schema)
    .format("csv")
    .option("header", "true")
    .load("abfss://bronze@stretaillakehouse01.dfs.core.windows.net/products/products_final.csv")
)

df_products.display()
df_products.printSchema()

Product_ID,Product_Name,Product_Category,Product_SubCategory,Product_Brand,Product_Price,Product_Unit,Product_Launch_Date
P0001,Wildcraft Backpack,Fashion,Wildcraft,Wildcraft,47511.43,Each,2024-01-22
P0002,Nike Sports T-Shirt,Fashion,Nike,Nike,34977.22,Each,2022-10-08
P0003,Prestige Pressure Cooker 5L,Home & Kitchen,Prestige,Prestige,49429.51,Set,2026-01-22
P0004,Tata Tea Gold,Grocery,Tata,Tata,24461.77,Litre,2023-02-19
P0005,Yoga Mat Pro,Sports & Fitness,Yoga,FitPro,43225.65,Each,2025-12-08
P0006,Milton Water Bottle,Home & Kitchen,Milton,Milton,13875.58,Set,2026-05-06
P0007,Boat Rockerz 450,Electronics,Boat,Boat,30877.01,Each,2023-10-09
P0008,Wildcraft Backpack,Fashion,Wildcraft,Wildcraft,15271.17,Pair,2024-08-22
P0009,Cello Storage Container,Home & Kitchen,Cello,Cello,7090.75,Set,2023-09-21
P0010,Studds Full Face Helmet,Vehicle Accessories,Studds,Studds,19247.52,Each,2024-09-22


root
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Product_SubCategory: string (nullable = true)
 |-- Product_Brand: string (nullable = true)
 |-- Product_Price: double (nullable = true)
 |-- Product_Unit: string (nullable = true)
 |-- Product_Launch_Date: date (nullable = true)



## Data Profiling

### Counting total number of products

In [0]:
df_product_total_count = df_products.count()
display(df_product_total_count)

1000

### Null value Analysis

In [0]:
df_products.columns

['Product_ID',
 'Product_Name',
 'Product_Category',
 'Product_SubCategory',
 'Product_Brand',
 'Product_Price',
 'Product_Unit',
 'Product_Launch_Date']

In [0]:
from pyspark.sql.functions import col, count, when

df_total_null_count= df_products.select(
    [
        count(when(col(c).isNull(), c)).alias(c)
        for c in df_products.columns
    ]
)
df_total_null_count.show()

+----------+------------+----------------+-------------------+-------------+-------------+------------+-------------------+
|Product_ID|Product_Name|Product_Category|Product_SubCategory|Product_Brand|Product_Price|Product_Unit|Product_Launch_Date|
+----------+------------+----------------+-------------------+-------------+-------------+------------+-------------------+
|         0|           0|               0|                  0|           14|            0|           0|                  0|
+----------+------------+----------------+-------------------+-------------+-------------+------------+-------------------+



### Duplicate value analysis

In [0]:
df_dup_product_count = df_products.groupBy("Product_ID").count()
df_dup_product_count = df_dup_product_count.filter(col("count") > 1)
df_dup_product_count.display()

Product_ID,count


### Business Rule validation

In [0]:
df_invalid_price_launch_date = df_products.filter((col("Product_Price")<=0) | 
                                       (col("Product_Launch_Date") > current_date()))

display(df_invalid_price_launch_date.limit(20))

Product_ID,Product_Name,Product_Category,Product_SubCategory,Product_Brand,Product_Price,Product_Unit,Product_Launch_Date


### Observation: No invalid product price and launch date were found during operation

## Data Standardization

In [0]:
df_trim_products = df_products\
                    .withColumn("Product_ID", trim(col("Product_ID")))\
                    .withColumn("Product_Name", trim(col("Product_Name")))\
                    .withColumn("Product_Category", trim(col("Product_Category")))\
                    .withColumn("Product_SubCategory", trim(col("Product_SubCategory")))\
                    .withColumn("Product_Brand", trim(col("Product_Brand")))\
                    .withColumn("Product_Unit", trim(col("Product_Unit")))
                           

                        

display(df_trim_products.limit(10))

Product_ID,Product_Name,Product_Category,Product_SubCategory,Product_Brand,Product_Price,Product_Unit,Product_Launch_Date
P0001,Wildcraft Backpack,Fashion,Wildcraft,Wildcraft,47511.43,Each,2024-01-22
P0002,Nike Sports T-Shirt,Fashion,Nike,Nike,34977.22,Each,2022-10-08
P0003,Prestige Pressure Cooker 5L,Home & Kitchen,Prestige,Prestige,49429.51,Set,2026-01-22
P0004,Tata Tea Gold,Grocery,Tata,Tata,24461.77,Litre,2023-02-19
P0005,Yoga Mat Pro,Sports & Fitness,Yoga,FitPro,43225.65,Each,2025-12-08
P0006,Milton Water Bottle,Home & Kitchen,Milton,Milton,13875.58,Set,2026-05-06
P0007,Boat Rockerz 450,Electronics,Boat,Boat,30877.01,Each,2023-10-09
P0008,Wildcraft Backpack,Fashion,Wildcraft,Wildcraft,15271.17,Pair,2024-08-22
P0009,Cello Storage Container,Home & Kitchen,Cello,Cello,7090.75,Set,2023-09-21
P0010,Studds Full Face Helmet,Vehicle Accessories,Studds,Studds,19247.52,Each,2024-09-22


### Writing in Silver Layer

In [0]:
(
    df_trim_products.write
        .format("parquet")
        .mode("overwrite")
        .save("abfss://silver@stretaillakehouse01.dfs.core.windows.net/products/")
)

### Reading for Silver Layer

In [0]:
df_silver_products = (
    spark.read
         .format("parquet")
         .load("abfss://silver@stretaillakehouse01.dfs.core.windows.net/products/")
)

### verifying for data loss

In [0]:
df_silver_products.count()

1000

In [0]:
df_products.count()

1000

In [0]:
df_silver_products.printSchema()

root
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Product_SubCategory: string (nullable = true)
 |-- Product_Brand: string (nullable = true)
 |-- Product_Price: double (nullable = true)
 |-- Product_Unit: string (nullable = true)
 |-- Product_Launch_Date: date (nullable = true)

